# FoundationalASSIST — xác minh schema, duplicate và missing

Đọc nguyên văn trường CSV, không tự đổi chuỗi NA thành missing. Không train, không tạo split, không ghi lại dữ liệu đã lọc. Chỉ hiển thị thống kê tổng hợp để tránh đưa ID học sinh hoặc câu trả lời vào Git.

FoundationalASSIST là dataset chính duy nhất trong workload mới; ASSIST09 chỉ là fallback. Các kết quả ASSIST09 đã có trong repo được giữ như lịch sử, không chạy thêm.


In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'scripts').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.audit_foundationalassist import audit
result = audit()
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "status": "read_only_audit_no_rules_locked_no_training",
  "provenance": {
    "Interactions": {
      "bytes": 245145636,
      "sha256": "671e97d320d0cf9b7e2bd75830d531cfbd95d307a1a9a590531934ad0d3d8ba4",
      "download_revision": "82b29188dffd2fd6bd3abc5a3de0db1ef1df12b9",
      "download_etag": "671e97d320d0cf9b7e2bd75830d531cfbd95d307a1a9a590531934ad0d3d8ba4"
    },
    "Problems": {
      "bytes": 1906331,
      "sha256": "37faf41fce19d6d228021ffeb74719db48be722f950ba413068aa3647f2e7e50",
      "download_revision": "82b29188dffd2fd6bd3abc5a3de0db1ef1df12b9",
      "download_etag": "4ab1a9093e6dd03af808c0fd5a8e6473c1de8e90"
    },
    "Skills": {
      "bytes": 256381,
      "sha256": "405c09530e767832cfc90e67a80153d20d0358503c40807b58e138557048cd68",
      "download_revision": "82b29188dffd2fd6bd3abc5a3de0db1ef1df12b9",
      "download_etag": "0bbadab5a0c201c913315858345d137e839915f1"
    }
  },
  "schema": {
    "Interactions": {
      "shape": [
        1753384,
        9


## Diễn giải và giới hạn

`Unnamed: 0` là cột được loại khỏi phép so sánh chẩn đoán; interaction `id` vẫn được giữ. Đối chiếu thêm đúng 7 trường nghiệp vụ và số ID có dữ liệu mâu thuẫn. Các phép loại trùng chỉ xảy ra trong bộ nhớ để đếm, chưa phải quy tắc preprocessing.

Revision lấy từ metadata tải Hugging Face, SHA-256 tính từ file hiện có. Chênh lệch 226 dòng với README chưa có giải thích xác thực. Không suy ra mất dữ liệu hoặc bù thêm dòng cho khớp README.

Cần xử lý metadata mâu thuẫn, lựa chọn quy tắc missing/multi-skill và kiểm tra bài phụ thuộc hình trước khi khóa protocol mới. Không sử dụng answer_text[t], hint_count[t], saw_answer[t] để dự đoán discrete_score[t].